In [1]:
import torch
from torch import nn
from d2l import torch as d2l


# ============================================================================
# 第1步：从零实现 dropout_layer 函数
# ============================================================================
# dropout_layer 是暂退法的核心函数，实现了标准的 Dropout 操作：
#   1. 生成与输入同形状的随机掩码（mask），元素服从均匀分布 U[0,1]
#   2. 将概率小于 p 的元素对应位置置零（即"丢弃"）
#   3. 将保留的元素除以 (1-p)，保持输出期望不变
def dropout_layer(X, dropout):
    # ---------------------------------------------------------------
    # assert：Python 断言语句，用于调试时检查条件。如果条件为 False，程序会抛出 AssertionError。
    #         这里确保暂退概率在 [0, 1] 合法范围内。
    # ---------------------------------------------------------------
    assert 0 <= dropout <= 1
    # ---------------------------------------------------------------
    # 特殊情况1：dropout == 1，丢弃全部神经元
    # torch.zeros_like(X)：返回一个与 X 形状相同、元素全为 0 的张量
    #                      （注意 "like" 表示形状和数据类型都与 X 一致）
    # ---------------------------------------------------------------
    if dropout == 1:
        return torch.zeros_like(X)
    # ---------------------------------------------------------------
    # 特殊情况2：dropout == 0，保留全部神经元（不做任何操作）
    # ---------------------------------------------------------------
    if dropout == 0:
        return X
    # ---------------------------------------------------------------
    # 核心逻辑：
    #   torch.rand(X.shape)：生成与 X 同形状的张量，每个元素独立地从均匀分布 U[0,1] 中随机采样
    #   > dropout：逐元素比较，生成布尔型张量（True/False）
    #             True 表示该位置被保留（随机数 > dropout），False 表示该位置被丢弃
    #   .float()：将布尔值转为浮点数（True→1.0, False→0.0），得到掩码张量 mask
    #
    # 形状变化：输入 X 形状为任意形状 → mask 形状与 X 完全相同
    # ---------------------------------------------------------------
    mask = (torch.rand(X.shape) > dropout).float()
    # ---------------------------------------------------------------
    # 逐元素乘法（Hadamard 积）：
    #   mask * X：保留的神经元保持原值，丢弃的神经元变为 0
    #   / (1.0 - dropout)：将保留的神经元放大，以保持该层输出的期望值不变
    #
    # 为什么除以 (1-p)？
    #   如果不除以 (1-p)，该层输出的期望值会变为原来的 (1-p) 倍。
    #   训练和测试时的数值尺度不一致，会导致性能下降。
    #   除以 (1-p) 后，E[输出] = E[X]，保证了无偏性。
    #
    # 示例：若 dropout=0.5，则保留的神经元值会乘以 2 (=1/(1-0.5))
    # ---------------------------------------------------------------
    return mask * X / (1.0 - dropout)


# ============================================================================
# 第2步：测试 dropout_layer 函数
# ============================================================================
# 使用 torch.arange 创建 0~15 的整数序列，重塑为 2×8 矩阵，便于观察 dropout 效果
# torch.arange(16, dtype=torch.float32)：生成 tensor([0., 1., 2., ..., 15.])
# .reshape((2, 8))：将一维向量重塑为 2 行 8 列的矩阵
# 输入形状：torch.Size([2, 8])
X = torch.arange(16, dtype=torch.float32).reshape((2, 8))
print("原始输入 X (2×8):")
print(X)
# ---------------------------------------------------------------
# 测试 dropout=0.0：不丢弃任何神经元，输出应与输入完全一致
# ---------------------------------------------------------------
print("\ndropout=0.0 (不丢弃任何神经元):")
print(dropout_layer(X, 0.))
# ---------------------------------------------------------------
# 测试 dropout=0.5：以 50% 概率丢弃神经元
#   被保留的元素值会变为原来的 2 倍（除以 1-0.5=0.5）
#   被丢弃的元素值变为 0
#   每次运行结果不同，因为掩码是随机生成的
# ---------------------------------------------------------------
print("\ndropout=0.5 (丢弃约一半神经元，保留的值放大2倍):")
print(dropout_layer(X, 0.5))
# ---------------------------------------------------------------
# 测试 dropout=1.0：丢弃所有神经元，输出全零矩阵
# ---------------------------------------------------------------
print("\ndropout=1.0 (丢弃全部神经元):")
print(dropout_layer(X, 1.))



原始输入 X (2×8):
tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.]])

dropout=0.0 (不丢弃任何神经元):
tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.]])

dropout=0.5 (丢弃约一半神经元，保留的值放大2倍):
tensor([[ 0.,  2.,  0.,  0.,  8.,  0.,  0.,  0.],
        [16., 18., 20.,  0., 24.,  0.,  0., 30.]])

dropout=1.0 (丢弃全部神经元):
tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])


In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(torch)